# Unidad 6 · Colab 1 de 3
## Estructura de repositorios y flujos de trabajo en GitHub

**Objetivos de este notebook**

- Diseñar la estructura de un repositorio profesional para una API o web app.
- Elegir y aplicar un flujo de trabajo con ramas (Git Flow, GitHub Flow, Trunk-Based Development).
- Practicar Pull Requests y buenas prácticas de code review.
- Escribir un `.gitignore` efectivo.
- Dar los primeros pasos con GitHub Actions (integración continua básica), que se retoma con más detalle en el Colab 3.

> **Nivel:** intermedio. Se asume que ya usás `git` y GitHub (clone, add, commit, push) y que contás con (o vas a construir) una API/web app de unidades anteriores para conectar con este contenido.

**Cómo usar este notebook:** cada sección tiene una explicación breve, links a la documentación oficial y uno o más ejercicios. Las soluciones están colapsadas debajo de cada ejercicio — intentalo primero y después comparálas.

---

## 1. Estructura de repositorios

Un repositorio bien organizado facilita que otras personas (y vos mismo en el futuro) entiendan, instalen y contribuyan al proyecto. No hay una única estructura correcta, pero sí elementos que casi siempre conviene tener:

| Archivo/carpeta | Para qué sirve |
|---|---|
| `README.md` | Qué hace el proyecto, cómo instalarlo y correrlo, cómo contribuir |
| `.gitignore` | Qué archivos no debe versionar Git (secretos, entornos virtuales, cachés) |
| `LICENSE` | Bajo qué términos se puede usar/reutilizar el código |
| `requirements.txt` / `pyproject.toml` | Dependencias del proyecto (Python) |
| `src/` o `app/` | Código fuente de la aplicación |
| `tests/` | Pruebas automatizadas |
| `docs/` | Documentación extendida |
| `.env.example` | Plantilla de variables de entorno (sin valores reales) |
| `.github/workflows/` | Workflows de GitHub Actions (CI/CD) |
| `Dockerfile`, `.dockerignore` | Contenedorización (Colab 2) |

Documentación oficial: [Acerca de los repositorios de GitHub](https://docs.github.com/es/repositories/creating-and-managing-repositories/about-repositories) · [Cómo escribir un buen README](https://docs.github.com/es/repositories/managing-your-repositorys-settings-and-features/customizing-your-repository/about-readmes)

### Ejercicio 1 — Diseñar la estructura de un repositorio

Vas a construir una **API de tareas (to-do) con FastAPI**, con tests, Docker y CI en GitHub Actions (todo lo que vas a ver en esta unidad). Escribí, en el siguiente bloque, el árbol de carpetas y archivos que usarías para este repositorio.

```text
mi-api-tareas/
(completá acá)
```

<details>
<summary>💡 Ver solución</summary>

```text
mi-api-tareas/
├── app/
│   ├── __init__.py
│   ├── main.py
│   ├── models.py
│   └── routes.py
├── tests/
│   └── test_routes.py
├── .github/
│   └── workflows/
│       └── ci.yml
├── .env.example
├── .gitignore
├── .dockerignore
├── Dockerfile
├── requirements.txt
├── LICENSE
└── README.md
```

</details>

## 2. Flujos de trabajo en GitHub

### Repaso rápido de comandos

```bash
git clone <url>
git checkout -b feature/mi-cambio
git add .
git commit -m 'Agrega endpoint de login'
git push -u origin feature/mi-cambio
```

Documentación oficial de Git: [git-scm.com/doc](https://git-scm.com/doc)

### Estrategias de branching

| Estrategia | Cómo funciona | Cuándo conviene |
|---|---|---|
| **GitHub Flow** | Una rama `main` siempre desplegable + ramas `feature/*` cortas que se mergean vía Pull Request | Proyectos con despliegue continuo, equipos chicos (ideal para esta unidad) |
| **Git Flow** | Ramas `main`, `develop`, `feature/*`, `release/*`, `hotfix/*` | Proyectos con versiones/releases planificados |
| **Trunk-Based Development** | Todos commitean directo (o con ramas muy cortas) a `main`, protegido con CI y feature flags | Equipos con alta madurez de CI/CD |

Más información: [GitHub Flow](https://docs.github.com/es/get-started/using-github/github-flow) · [Git Flow (Vincent Driessen)](https://nvie.com/posts/a-successful-git-branching-model/) · [Trunk Based Development](https://trunkbaseddevelopment.com/)

Para esta unidad vamos a usar **GitHub Flow**: es el más simple y es el que mejor combina con el despliegue continuo en Render/Railway/HF Spaces que vas a ver en el Colab 3.

### Pull Requests y code review

Un Pull Request (PR) propone fusionar una rama en otra y habilita revisión de código antes del merge. Buenas prácticas: PRs chicos y enfocados, descripción clara de qué cambia y por qué, y al menos una revisión aprobada antes de mergear.

Documentación oficial: [Acerca de los pull requests](https://docs.github.com/es/pull-requests/collaborating-with-pull-requests/proposing-changes-to-your-work-with-pull-requests/about-pull-requests)

In [1]:
# Practicá el flujo de feature branch en un repo de prueba (esto SÍ corre en Google Colab)
!rm -rf /content/demo-repo
!mkdir -p /content/demo-repo && cd /content/demo-repo && git init -q
!cd /content/demo-repo && git config user.email 'alumno@example.com' && git config user.name 'Alumno'
!cd /content/demo-repo && echo '# Demo' > README.md && git add . && git commit -q -m 'Commit inicial'

# TODO Ejercicio 2: completá los pasos que faltan
# 1) Crear y moverte a una rama feature/login
# 2) Crear un archivo login.py con cualquier contenido
# 3) Agregar y commitear el cambio en esa rama
# 4) Volver a main y mergear feature/login con --no-ff
# 5) Mostrar el log con --oneline --graph --all

<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```bash
cd /content/demo-repo
git checkout -b feature/login
echo 'def login(): pass' > login.py
git add login.py
git commit -m 'Agrega función de login'
git checkout main
git merge --no-ff feature/login -m 'Merge feature/login'
git log --oneline --graph --all
```

</details>

In [3]:
import os
import re
import sqlite3
from bs4 import BeautifulSoup
import pandas as pd

# -------------------------------------------------------------
# 1. Asegurar que el archivo HTML exista (lo crea si no existe)
# -------------------------------------------------------------
HTML_PATH = "competidores.html"

if not os.path.exists(HTML_PATH):
    html_content = """<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <title>Tienda Competidor</title>
</head>
<body>
    <h1>Catálogo Oficial - Tienda Competidor</h1>
    <div class="catalogo">
        <div class="producto">
            <h2 class="nombre">Mouse Inalámbrico Logitech</h2>
            <span class="categoria">Tecnología</span>
            <p class="precio-anterior">$18.500</p>
            <p class="precio-actual">$12.999</p>
        </div>
        <div class="producto">
            <h2 class="nombre">Teclado Mecánico RGB</h2>
            <span class="categoria">Tecnología</span>
            <p class="precio-anterior">$32.000</p>
            <p class="precio-actual">$24.500</p>
        </div>
        <div class="producto">
            <h2 class="nombre">Lámpara LED Escritorio</h2>
            <span class="categoria">Hogar</span>
            <p class="precio-actual">$9.450</p>
        </div>
        <div class="producto">
            <h2 class="nombre">Auriculares Bluetooth</h2>
            <span class="categoria">Tecnología</span>
            <p class="precio-anterior">$19.999</p>
            <p class="precio-actual">$14.200</p>
        </div>
        <div class="producto">
            <h2 class="nombre">Cafetera de Filtro Eléctrica</h2>
            <span class="categoria">Hogar</span>
            <p class="precio-anterior">$45.000</p>
            <p class="precio-actual">$38.900</p>
        </div>
    </div>
</body>
</html>"""
    with open(HTML_PATH, "w", encoding="utf-8") as f:
        f.write(html_content)

# -------------------------------------------------------------
# 2. Extracción con BeautifulSoup
# -------------------------------------------------------------
with open(HTML_PATH, "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f, "html.parser")

registros = []
tarjetas = soup.select(".producto")

for t in tarjetas:
    tag_nombre = t.select_one(".nombre")
    tag_categoria = t.select_one(".categoria")
    tag_actual = t.select_one(".precio-actual")
    tag_anterior = t.select_one(".precio-anterior")

    registros.append(
        {
            "nombre": tag_nombre.get_text(strip=True) if tag_nombre else None,
            "categoria": (
                tag_categoria.get_text(strip=True) if tag_categoria else None
            ),
            "precio_actual_raw": (
                tag_actual.get_text(strip=True) if tag_actual else None
            ),
            "precio_anterior_raw": (
                tag_anterior.get_text(strip=True) if tag_anterior else None
            ),
        }
    )

df = pd.DataFrame(registros)


# -------------------------------------------------------------
# 3. Limpieza y conversión a numérico (float)
# -------------------------------------------------------------
def a_float(valor):
    if not valor or pd.isna(valor):
        return None
    # Remueve signos monetarios, espacios y separa decimales
    limpio = re.sub(r"[^\d,.-]", "", str(valor))
    limpio = limpio.replace(".", "").replace(",", ".")
    try:
        return float(limpio)
    except ValueError:
        return None


df["precio_actual"] = df["precio_actual_raw"].apply(a_float)
df["precio_anterior"] = df["precio_anterior_raw"].apply(a_float)

df_final = df[
    ["nombre", "categoria", "precio_actual", "precio_anterior"]
].copy()

# -------------------------------------------------------------
# 4. Guardado en SQLite (competidores_int.db)
# -------------------------------------------------------------
DB_NAME = "competidores_int.db"
conn = sqlite3.connect(DB_NAME)

df_final.to_sql("precios_competencia", conn, if_exists="replace", index=False)
print(f"Base de datos '{DB_NAME}' actualizada con {len(df_final)} registros.\n")

# -------------------------------------------------------------
# 5. Consulta SQL con Pandas (< 15000)
# -------------------------------------------------------------
query = """
SELECT
    nombre,
    categoria,
    precio_actual,
    precio_anterior
FROM precios_competencia
WHERE precio_actual < 15000
ORDER BY precio_actual ASC;
"""

df_resultado = pd.read_sql(query, conn)
conn.close()

# Mostrar la tabla filtrada en Colab
display(df_resultado)

Base de datos 'competidores_int.db' actualizada con 5 registros.



,nombre,categoria,precio_actual,precio_anterior
0,Lámpara LED Escritorio,Hogar,9450.0,NaN
1,Mouse Inalámbrico Logitech,Tecnología,12999.0,18500.0
2,Auriculares Bluetooth,Tecnología,14200.0,19999.0


### `.gitignore`

Evita versionar archivos que no deberían estar en el repositorio: entornos virtuales, cachés, credenciales, artefactos de build.

Ejemplo típico para un proyecto Python + Docker:

```gitignore
__pycache__/
*.pyc
.venv/
env/
.env
.DS_Store
*.log
.pytest_cache/
```

Plantillas oficiales: [github/gitignore](https://github.com/github/gitignore) · generador: [gitignore.io](https://www.toptal.com/developers/gitignore)

### Ejercicio 3 — Escribir un `.gitignore`

Tu proyecto combina un backend en **Python** (FastAPI), un frontend en **Node** (React) y se dockeriza. Escribí un `.gitignore` que cubra los tres casos (Python, Node y archivos de tu editor/IDE).

```gitignore
(completá acá)
```

<details>
<summary>💡 Ver solución</summary>

```gitignore
# Python
__pycache__/
*.pyc
.venv/
.env

# Node
node_modules/
dist/
npm-debug.log*

# Docker
*.pid

# Editor / SO
.vscode/
.idea/
.DS_Store
```

</details>

In [5]:
%%writefile .gitignore
# Python
__pycache__/
*.pyc
.venv/
.env

# Node
node_modules/
dist/
npm-debug.log*

# Docker
*.pid

# Editor / SO
.vscode/
.idea/
.DS_Store

Writing .gitignore


## 3. Introducción a GitHub Actions (CI básica)

GitHub Actions permite automatizar tareas (correr tests, lint, build, deploy) cada vez que ocurre un evento en el repo, como un `push` o un `pull_request`. Un workflow se define en un archivo YAML dentro de `.github/workflows/`.

Conceptos clave: **workflow** (el archivo completo), **evento disparador** (`on:`), **job** (conjunto de pasos que corren en una máquina), **step** (una acción o comando individual), **runner** (la máquina virtual, ej. `ubuntu-latest`).

```yaml
name: CI

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      - run: pip install -r requirements.txt
      - run: pytest
```

Documentación oficial: [Guía de inicio rápido de GitHub Actions](https://docs.github.com/es/actions/writing-workflows/quickstart) · [Sintaxis de workflows](https://docs.github.com/es/actions/writing-workflows/workflow-syntax-for-github-actions)

En el Colab 3 vas a conectar este workflow con el despliegue automático en Render/Railway/HF Spaces.

### Ejercicio 4 — Escribir un workflow de CI

Tu API de tareas usa **FastAPI**, sus tests están en la carpeta `tests/` y corren con `pytest`. Escribí `.github/workflows/ci.yml` para que, en cada `push` y `pull_request` a `main`, se instale Python 3.11, se instalen las dependencias y se corran los tests.

```yaml
(completá acá)
```

<details>
<summary>💡 Ver solución</summary>

```yaml
name: CI

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      - run: pip install -r requirements.txt
      - run: pytest -v
```

</details>

In [6]:
!mkdir -p .github/workflows

In [7]:
%%writefile .github/workflows/ci.yml
name: CI

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      - run: pip install -r requirements.txt
      - run: pytest -v

Writing .github/workflows/ci.yml


In [8]:
!cat .github/workflows/ci.yml

name: CI

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      - run: pip install -r requirements.txt
      - run: pytest -v


## Mini-proyecto integrador

Sobre tu propio repositorio (o uno nuevo de práctica):

1. Organizá la estructura de carpetas siguiendo la tabla de la sección 1.
2. Creá una rama `feature/algo`, hacé un cambio y abrí un Pull Request hacia `main` (podés hacerlo con un repo tuyo en GitHub, no solo local).
3. Escribí un `.gitignore` adecuado a tu stack.
4. Agregá un workflow `.github/workflows/ci.yml` que corra tus tests.
5. Verificá en la pestaña **Actions** de GitHub que el workflow se ejecuta en verde.

**Entregable:** link al repositorio en GitHub con el PR (mergeado o abierto) y el workflow corriendo en verde.

---

**Seguís en:** *Colab 2 — Introducción a la contenedorización: Docker*

In [11]:
import os
import subprocess

# 1. Crear la estructura de directorios
os.makedirs("src", exist_ok=True)
os.makedirs("tests", exist_ok=True)
os.makedirs(".github/workflows", exist_ok=True)

# 2. Archivo .gitignore
gitignore_content = """# Python
__pycache__/
*.py[cod]
*$py.class
*.so
.Python
env/
venv/
.venv/
.env

# Pruebas y cobertura
.pytest_cache/
.coverage
htmlcov/

# IDEs y sistemas operativos
.vscode/
.idea/
*.swp
.DS_Store
"""
with open(".gitignore", "w", encoding="utf-8") as f:
    f.write(gitignore_content)

# 3. Archivo requirements.txt
requirements_content = """fastapi>=0.110.0
pydantic>=2.6.0
pytest>=8.0.0
httpx>=0.27.0
"""
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements_content)

# 4. Archivo src/app.py
app_content = """from datetime import datetime
from fastapi import FastAPI, status
from pydantic import BaseModel, Field

app = FastAPI(
    title="API Integrador CI/CD",
    version="1.0.0",
    description="Microservicio base para validación de pipelines automatizados con GitHub Actions."
)

class ItemInput(BaseModel):
    nombre: str = Field(..., min_length=2, description="Nombre del ítem")
    precio: float = Field(..., gt=0, description="Precio unitario mayor a cero")

@app.get("/health", tags=["Monitoreo"], status_code=status.HTTP_200_OK)
def health_check():
    return {"status": "ok", "timestamp": datetime.utcnow().isoformat()}

@app.post("/items", tags=["Operaciones"], status_code=status.HTTP_201_CREATED)
def crear_item(item: ItemInput):
    precio_con_iva = round(item.precio * 1.21, 2)
    return {
        "nombre": item.nombre,
        "precio_base": item.precio,
        "precio_final": precio_con_iva,
        "procesado": True
    }

@app.get("/descuento", tags=["Operaciones"], status_code=status.HTTP_200_OK)
def calcular_descuento(precio: float, porcentaje: float = 10.0):
    descuento = precio * (porcentaje / 100.0)
    return {
        "precio_original": precio,
        "descuento_aplicado": round(descuento, 2),
        "precio_descuento": round(precio - descuento, 2)
    }
"""
with open("src/app.py", "w", encoding="utf-8") as f:
    f.write(app_content)

# Archivos __init__.py
open("src/__init__.py", "w").close()
open("tests/__init__.py", "w").close()

# 5. Archivo tests/test_app.py
test_content = """from fastapi import status
from fastapi.testclient import TestClient
from src.app import app

client = TestClient(app)

def test_health_check():
    res = client.get("/health")
    assert res.status_code == status.HTTP_200_OK
    assert res.json()["status"] == "ok"

def test_crear_item_valido():
    payload = {"nombre": "Teclado", "precio": 100.0}
    res = client.post("/items", json=payload)
    assert res.status_code == status.HTTP_201_CREATED
    data = res.json()
    assert data["nombre"] == "Teclado"
    assert data["precio_final"] == 121.0
    assert data["procesado"] is True

def test_crear_item_precio_invalido():
    payload = {"nombre": "Mouse", "precio": -10.0}
    res = client.post("/items", json=payload)
    assert res.status_code == status.HTTP_422_UNPROCESSABLE_ENTITY

def test_calcular_descuento():
    res = client.get("/descuento?precio=200&porcentaje=15")
    assert res.status_code == status.HTTP_200_OK
    assert res.json()["precio_descuento"] == 170.0
"""
with open("tests/test_app.py", "w", encoding="utf-8") as f:
    f.write(test_content)

# 6. Archivo .github/workflows/ci.yml
ci_workflow_content = """name: CI

on:
  push:
    branches: [ main ]
  pull_request:
    branches: [ main ]

jobs:
  test:
    runs-on: ubuntu-latest

    steps:
      - name: Descargar código del repositorio
        uses: actions/checkout@v4

      - name: Configurar entorno de Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'
          cache: 'pip'

      - name: Instalar dependencias
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt

      - name: Ejecutar tests con pytest
        run: |
          pytest -v
"""
with open(".github/workflows/ci.yml", "w", encoding="utf-8") as f:
    f.write(ci_workflow_content)

# 7. Archivo README.md (con formato limpio sin colisión de bloques)
readme_content = (
    "# Microservicio Integrador CI/CD\n\n"
    "API construida con FastAPI, testeada con pytest y validada automáticamente "
    "en cada Push o Pull Request hacia main mediante GitHub Actions.\n\n"
    "## Estructura\n"
    "- src/app.py: Código principal de la API.\n"
    "- tests/test_app.py: Suite de tests automatizados.\n"
    "- .github/workflows/ci.yml: Definición del pipeline CI.\n\n"
    "## Ejecución local\n"
    "pip install -r requirements.txt\n"
    "pytest -v\n"
)
with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

print("Archivos y carpetas del proyecto generados exitosamente.")

# 8. Instalación de dependencias y ejecución de tests locales
print("\nInstalando dependencias y corriendo pytest...")
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run(["pytest", "-v"], check=True)

Archivos y carpetas del proyecto generados exitosamente.

Instalando dependencias y corriendo pytest...


CompletedProcess(args=['pytest', '-v'], returncode=0)